Part 1: TabNet Framework for Google Colab

TabNet uses pytorch-tabnet which is fully integrated with Scikit-Learn-like structures (fit, predict, predict_proba), making it incredibly straightforward to implement.

Step 1: Install Dependencies & Setup Mock Data

Run this block in a cell to install the required libraries and simulate a subset of your 3,924 high-dimensional dataset.

In [1]:
# Cell 1: Setup and Data Generation
!pip install pytorch-tabnet

import numpy as np
import pandas as pd
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc

# --- Simulate Hackathon Data Layout ---
np.random.seed(42)
num_samples = 5000
num_features = 200 # Using top 200 variance features as an optimized space

# Create mock data for features F1 to F199, and specific bank features
feature_names = [f"F{i}" for i in range(1, num_features)] + ['F115', 'F321', 'F527', 'F3894']
X_mock = np.random.randn(num_samples, len(feature_names))

# Create a highly imbalanced target column (feature_3924) where 1 = Mule, 0 = Legitimate
# 2% true fraud rate
y_mock = np.random.choice([0, 1], size=num_samples, p=[0.98, 0.02])

# Build complete dataframe
df = pd.DataFrame(X_mock, columns=feature_names)
df['feature_3924'] = y_mock

print(f"Data balance check:\n{df['feature_3924'].value_counts(normalize=True)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.8 MB/s eta 0:00:00
Data balance check:
feature_3924
0    0.9788
1    0.0212
Name: proportion, dtype: float64


Step 2: Preprocessing & Data Splits

Prepare your feature space arrays and handle the explicit class imbalances.

In [2]:
# Cell 2: Train-test Splitting
X = df.drop(columns=['feature_3924']).values
y = df['feature_3924'].values

# Stratified split to ensure the 2% fraud cases are cleanly distributed
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Calculate ratio to handle class imbalance (mules are rare)
counts = np.bincount(y_train)
pos_weight = counts[0] / counts[1]
print(f"Calculated weight adjustments for imbalance: {pos_weight:.2f}")

Calculated weight adjustments for imbalance: 46.06


Step 3: Model Training & Feature Masks (Explainable AI)

We train the TabNetClassifier. One massive perk of TabNet is that it outputs structural masks showing which features it examined at each validation phase.

In [4]:
# Cell 3: Training TabNet
tabnet_model = TabNetClassifier(
    n_d=16, n_a=16, n_steps=4,           # Architecture deep specs
    gamma=1.3,                           # Relaxing factor for feature reuse
    lambda_sparse=1e-3,                  # Sparsity loss weight
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='sparsemax',               # Softmax forcing sparse selections
    device_name='cuda' if torch.cuda.is_available() else 'cpu'
)

# Fit the classifier
tabnet_model.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_val, y_val)],
    eval_name=['valid'],
    eval_metric=['auc'],
    max_epochs=30,
    patience=10,
    batch_size=256, virtual_batch_size=64, # Ghost Batch Normalization sizing
    drop_last=False,
    weights={0: 1, 1: pos_weight}                # Handles class imbalance
)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.88079 | valid_auc: 0.61005 |  0:00:01s
epoch 1  | loss: 0.68731 | valid_auc: 0.58729 |  0:00:02s
epoch 2  | loss: 0.63189 | valid_auc: 0.41772 |  0:00:03s
epoch 3  | loss: 0.60812 | valid_auc: 0.41578 |  0:00:04s
epoch 4  | loss: 0.5585  | valid_auc: 0.48957 |  0:00:05s
epoch 5  | loss: 0.4853  | valid_auc: 0.51097 |  0:00:06s
epoch 6  | loss: 0.45175 | valid_auc: 0.59794 |  0:00:07s
epoch 7  | loss: 0.41975 | valid_auc: 0.57114 |  0:00:08s
epoch 8  | loss: 0.36466 | valid_auc: 0.45586 |  0:00:10s
epoch 9  | loss: 0.33949 | valid_auc: 0.48023 |  0:00:11s
epoch 10 | loss: 0.29684 | valid_auc: 0.4937  |  0:00:12s

Early stopping occurred at epoch 10 with best_epoch = 0 and best_valid_auc = 0.61005


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Step 4: Model Evaluation

Evaluate utilizing Precision-Recall AUC (PR-AUC) which focuses exactly on catching the rare positive class without boosting score distortions.

In [5]:
# Cell 4: Evaluate Model
preds_proba = tabnet_model.predict_proba(X_val)[:, 1]
preds = tabnet_model.predict(X_val)

precision, recall, _ = precision_recall_curve(y_val, preds_proba)
pr_auc = auc(recall, precision)

print(f"\n Validation PR-AUC: {pr_auc:.4f}")
print("\nClassification Matrix Metrics:")
print(classification_report(y_val, preds))

# Extract explainable feature importances mapped by TabNet attention mechanisms
importance = tabnet_model.feature_importances_
feat_imp_df = pd.DataFrame({'Feature': df.drop(columns=['feature_3924']).columns, 'Importance': importance})
print("\nTop 5 Drivers of Fraud Flagging via TabNet Attention:")
print(feat_imp_df.sort_values(by='Importance', ascending=False).head(5))


 Validation PR-AUC: 0.0861

Classification Matrix Metrics:
              precision    recall  f1-score   support

           0       0.98      0.58      0.73       979
           1       0.03      0.57      0.05        21

    accuracy                           0.58      1000
   macro avg       0.51      0.58      0.39      1000
weighted avg       0.96      0.58      0.72      1000


Top 5 Drivers of Fraud Flagging via TabNet Attention:
    Feature  Importance
120    F121    0.024630
68      F69    0.017841
124    F125    0.017706
170    F171    0.016614
59      F60    0.016063


Part 2: Graph Neural Network (GNN) Framework for Google Colab

For account security, mapping the transaction infrastructure as a graph yields better structural results than standard tabular processing. We'll use PyTorch Geometric (PyG) to build a graph classification workflow.

Step 1: Install PyTorch Geometric Dependencies

PyG relies on complex system-level binaries. Execute this precise installation routine inside Colab.

In [6]:
# Cell 5: Dynamic PyG Installation
import torch
print(f"PyTorch Version: {torch.__version__}")

# Fetch matching compiled wheel paths cleanly
TORCH_version = torch.__version__.split('+')[0]
os_env = "cu121" if torch.cuda.is_available() else "cpu" # Defaulting configurations

!pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH_version}+{os_env}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{TORCH_version}+{os_env}.html
!pip install torch-geometric

PyTorch Version: 2.11.0+cpu
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.6/681.6 kB 6.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.2 MB/s eta 0:00:00


Step 2: Build a Transactional Graph Object

We will transform account rows and connection paths into a single torch_geometric.data.Data graph network.

In [7]:
# Cell 6: Create Graph Infrastructure
import torch
from torch_geometric.data import Data

num_accounts = 1000
num_features_per_account = 18 # Map our 18 core bank features (F115, F321, etc)

# 1. Node Feature Matrix (X): Shape [num_nodes, num_features]
node_features = torch.randn((num_accounts, num_features_per_account), dtype=torch.float)

# 2. Target Variables (Y): Classify each individual node/account (0=Legit, 1=Mule)
node_labels = torch.from_numpy(np.random.choice([0, 1], size=num_accounts, p=[0.95, 0.05])).long()

# 3. Graph Layout Edges (edge_index): Transactions passing between source -> target indices
# Generating 3000 random transacting links between accounts
source_nodes = np.random.randint(0, num_accounts, size=3000)
target_nodes = np.random.randint(0, num_accounts, size=3000)
edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)

# Create masks for evaluation within a single continuous graph structure
train_mask = torch.zeros(num_accounts, dtype=torch.bool)
val_mask = torch.zeros(num_accounts, dtype=torch.bool)
train_mask[:700] = True
val_mask[700:900] = True

# Bundle into PyG data container
graph_data = Data(x=node_features, edge_index=edge_index, y=node_labels)
graph_data.train_mask = train_mask
graph_data.val_mask = val_mask

print("Graph Summary Details:")
print(f"Number of Bank Accounts (Nodes): {graph_data.num_nodes}")
print(f"Number of Transactions (Edges): {graph_data.num_edges}")

Graph Summary Details:
Number of Bank Accounts (Nodes): 1000
Number of Transactions (Edges): 3000


/tmp/ipykernel_423/1646955667.py:18: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)


Step 3: Define a GCN (Graph Convolutional Network) Architecture

This network aggregates features from neighboring accounts. If an account interacts heavily with other flagged mule nodes, its own risk score increases via neighborhood message-passing.

In [8]:
# Cell 7: Define GCN Model
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import Linear

class MuleGCN(torch.nn.Module):
    def __init__(self, num_node_features, num_classes):
        super(MuleGCN, self).__init__()
        # 2-Layer Graph Convolutional Engine
        self.conv1 = GCNConv(num_node_features, 32)
        self.conv2 = GCNConv(32, 16)
        self.classifier = Linear(16, num_classes)

    def forward(self, x, edge_index):
        # First layer with Relu and Dropout to counter overfitting
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)

        # Second layer aggregation
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Output Logits
        logits = self.classifier(x)
        return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MuleGCN(num_node_features=num_features_per_account, num_classes=2).to(device)
graph_data = graph_data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

print(model)

MuleGCN(
  (conv1): GCNConv(18, 32)
  (conv2): GCNConv(32, 16)
  (classifier): Linear(in_features=16, out_features=2, bias=True)
)


Step 4: GNN Training & Inference


Run the neighborhood optimization epochs.

In [9]:
# Cell 8: GNN Optimization Loop
def train_gnn():
    model.train()
    optimizer.zero_grad()

    # Forward pass calculates states for ALL nodes across the whole network graph
    out = model(graph_data.x, graph_data.edge_index)

    # Calculate CrossEntropyLoss using only the training slice mask
    loss = F.cross_entropy(out[graph_data.train_mask], graph_data.y[graph_data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate_gnn():
    model.eval()
    out = model(graph_data.x, graph_data.edge_index)
    probabilities = F.softmax(out, dim=1)[:, 1] # Extract true risk probabilities

    # Extract validation targets and predictions
    val_targets = graph_data.y[graph_data.val_mask].cpu().numpy()
    val_scores = probabilities[graph_data.val_mask].cpu().numpy()
    val_predictions = val_scores >= 0.5

    precision, recall, _ = precision_recall_curve(val_targets, val_scores)
    return auc(recall, precision), val_targets, val_predictions

# Run Training Loops
print("Training Graph Neural Network...")
for epoch in range(1, 51):
    loss = train_gnn()
    if epoch % 10 == 0:
        pr_auc, _, _ = evaluate_gnn()
        print(f"Epoch {epoch:02d} | Train Loss: {loss:.4f} | Val PR-AUC: {pr_auc:.4f}")

# Final Metric Outputs
final_auc, targets, binary_preds = evaluate_gnn()
print(f"\n Final GNN Node Classification PR-AUC: {final_auc:.4f}")
print("\nGraph Classification Report:")
print(classification_report(targets, binary_preds))

Training Graph Neural Network...
Epoch 10 | Train Loss: 0.1768 | Val PR-AUC: 0.0231
Epoch 20 | Train Loss: 0.1800 | Val PR-AUC: 0.0207
Epoch 30 | Train Loss: 0.1423 | Val PR-AUC: 0.0208
Epoch 40 | Train Loss: 0.1365 | Val PR-AUC: 0.0238
Epoch 50 | Train Loss: 0.1299 | Val PR-AUC: 0.0271

 Final GNN Node Classification PR-AUC: 0.0271

Graph Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       195
           1       0.00      0.00      0.00         5

    accuracy                           0.97       200
   macro avg       0.49      0.50      0.49       200
weighted avg       0.95      0.97      0.96       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
